[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shulruferor-sudo/Advance-Data-Science/blob/main/projects/Bureau%20of%20Statistics/Assignment_Solution.ipynb)


# Occupational Classification: Survey Data and Model Evaluation

This notebook examines survey data and metadata, tests field-specific preparation decisions, and evaluates occupational classification models. Each transformation and modeling decision is tied to observable evidence and assessed against its alternatives.


In [2]:
# Standard library and notebook-wide utilities
import io
import hashlib
import urllib.request
import re
import json
import warnings
from pathlib import Path
import importlib.util
import subprocess
import sys
from urllib.request import urlopen


# Keep the previous warning policy scoped to UserWarning, not all warnings.
warnings.filterwarnings('ignore', category=UserWarning)

# Numerical arrays and tabular survey data
import numpy as np
import pandas as pd

# Notebook display configuration: improve readability without changing data.
pd.set_option('display.html.use_mathjax', False)
pd.set_option('display.max_columns', None)

def custom_style(df):
    """Format DataFrames for notebook display; source values remain unchanged."""
    return (df.style.set_table_styles([
        {'selector': 'th', 'props': [
            ('background-color', '#1e293b'), ('color', '#ffffff'),
            ('font-family', 'sans-serif'), ('font-size', '13px'),
            ('padding', '10px 15px'), ('text-align', 'left')
        ]},
        {'selector': 'td', 'props': [
            ('font-family', 'sans-serif'), ('font-size', '12px'),
            ('padding', '8px 15px'), ('border-bottom', '1px solid #e2e8f0')
        ]},
        {'selector': 'tr:hover', 'props': [('background-color', '#f1f5f9')]}
    ]).set_properties(**{'background-color': '#f8fafc'}))

# Apply the HTML style to all DataFrame previews in this notebook session.
pd.DataFrame._repr_html_ = lambda self: custom_style(self)._repr_html_()

# Figures and rich notebook output
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Machine learning: cross-validation and reusable pipelines
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline

# Candidate classifiers and majority-class baseline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.dummy import DummyClassifier

# Performance measurement
from sklearn.metrics import accuracy_score, f1_score, classification_report

# Reproducibility: downstream model evaluation uses this fixed random seed.
SEED = 42


### Load the LaBSE model

[LaBSE](https://huggingface.co/sentence-transformers/LaBSE?utm_source=chatgpt.com)

LaBSE (Language-agnostic BERT Sentence Embedding) is a multilingual model that converts text into 768-dimensional, L2-normalized vectors in a shared space. It was optimized to place translations close together, so we use it to compare workplace names written in Hebrew and English without translating them first. Because the vectors have unit length, their dot product equals cosine similarity. A high score identifies a candidate match between two workplace names. Establishing that both names refer to the same employer requires corroborating information.

The code below loads the pretrained model on the CPU. In the workplace-name analysis, we use cosine similarity to rank possible Hebrew matches for names written in English. A pair goes forward for review when its cosine similarity exceeds the adjustable threshold α. Confirming that both names refer to the same workplace requires separate evidence. 


In [31]:
# import sentence-transformers if not already installed
if importlib.util.find_spec("sentence_transformers") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "sentence-transformers==5.1.2", ])
from sentence_transformers import SentenceTransformer

# Download/cache LaBSE once. No API token, SSH, or paid GPU.
model = SentenceTransformer("sentence-transformers/LaBSE", device="cpu")
model.eval()
print("LaBSE loaded.")

LaBSE loaded.


## Source data

Load the survey responses and their metadata from the two workbook sheets without altering the source values.


In [32]:
# Retrieve the original workbook, verify its contents, and load both source tables.
URL = "https://raw.githubusercontent.com/shulruferor-sudo/Advance-Data-Science/main/Datasets/cbs_occupation_survey_sample/DUMMY_DATASET.xlsx"
SHA = "e1e7f391c0675dd14830eaae6b5cd114edc256f18b956bb48a454b0b1209e1ac"
raw = urllib.request.urlopen(URL, timeout=90).read()

# Load only rows and columns without missing values
data = (pd.read_excel(io.BytesIO(raw), sheet_name="Survey Dataset").dropna(axis=0, how="all").dropna(axis=1, how="all"))

metadata = (pd.read_excel(io.BytesIO(raw), sheet_name="Metadata").dropna(axis=0, how="all").dropna(axis=1, how="any"))

# Display the complete metadata dictionary.
display(metadata)
print()

# Display the first three survey records.
display(data.head(20))
print()

,משתנה,הסבר,פורמט
0,ID,מספר רץ,INTEGER
1,YeshuvAvoda,שם יישוב עבודה,TEXT
2,ShemAvoda,שם מקום העבודה שלך,TEXT
3,SugAvoda,הפעילות העיקרית של מקום העבודה,TEXT
4,ShemMachlaka,אגף או מחלקה שהנסקר עובד,TEXT
5,SugMachlaka,הפעילות העיקרית של האגף או המחלקה שהנסקר עובד,TEXT
6,EzoAvoda,עבודה שהנסקר עושה במקום עבודה זה,TEXT
7,TeurPeula,הפעולות העיקריות שהנסקר עושה במקום עבודה,TEXT
8,TeurTafkid,מהו תואר המשרה של הנסקר במקום עבודה זה (במידה ויש),TEXT
9,MenahelEtMi,האם הנסקר מנהל או ממונה על עובדים אחרים?,INTEGER


,ID,YeshuvAvoda,ShemAvoda,SugAvoda,ShemMachlaka,SugMachlaka,EzoAvoda,TeurPeula,TeurTafkid,MenahelEtMi,MaamadAvoda,MakorSachar,TeudaGvoha,shnotlimud,Gil,TarichHathala,SemelAnafSofi,SemelMishlachSofi
0,1.000000,ירושלים,"למ""ס","איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו,סגן מנהל אגף,"כנ""ל",סגן מנהל אגף,3.000000,1.000000,1.000000,6.000000,22.000000,55.000000,2020-06-15 00:00:00,8411,2120
1,2.000000,ירושלים,בית חולים הדסה עין כרם,טיפול בחולים,רדיואתרפיה,טיפול בחולי סרטן,טכנאית רדיותרפיה,"כנ""ל","כנ""ל",nan,1.000000,1.000000,6.000000,22.000000,55.000000,2022-07-01 00:00:00,8610,3311
2,3.000000,באר שבע,קריית הממשלה,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל",נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט,nan,nan,1.000000,1.000000,5.000000,15.000000,29.000000,2021-06-06 00:00:00,8411,3512
3,4.000000,תל אביב,משרד החינוך,בית ספר יסודי,שמירה,nan,שומר בבית הספר,nan,nan,nan,1.000000,2.000000,3.000000,12.000000,48.000000,2024-07-01 00:00:00,8010,5414
4,5.000000,תל אביב,אמיר כהן,מטפל בקשיש במשפחה,nan,nan,מטפל בקשיש,"נותן אוכל, עוזר בניקיון בית, עוזר להתרחץ",nan,nan,1.000000,1.000000,4.000000,14.000000,42.000000,2010-01-11 00:00:00,XXXX,5322
5,6.000000,חיפה,משטרה,שוטר,nan,nan,שוטר,nan,nan,nan,1.000000,1.000000,4.000000,14.000000,35.000000,2022-05-31 00:00:00,8423,5412
6,7.000000,קרית גת,אינטל,ייצור צ'יפים למחשבים,חדר נקי,nan,עובד בחדר נקי,מפקח על תהליך ייצור צ'יפים,nan,nan,1.000000,1.000000,5.000000,15.000000,65.000000,1999-07-07 00:00:00,2610,3122
7,8.000000,חיפה,אינטל,פיתוח תוכנה לצ'יפים,הנדסת תוכנה,nan,"מהנדס תוכנה, מנהל צוות עובדים",מפתח קוד לצ'יפים,nan,1.000000,1.000000,1.000000,6.000000,18.000000,55.000000,2000-11-10 00:00:00,7201,2512
8,9.000000,אשקלון,שופר סל,קופאית,nan,nan,קופאית,nan,nan,nan,1.000000,3.000000,2.000000,11.000000,52.000000,2022-11-11 00:00:00,4711,5230
9,10.000000,הרצליה,EMILIO,"הייטק, פיתוח תוכנה",nan,nan,מתכנת,nan,nan,nan,1.000000,1.000000,5.000000,16.000000,24.000000,2021-12-14 00:00:00,6201,2514


### Field-by-field investigation


### Analysis of ID column

The metadata defines the values within the column ID as a values of a running index.
The metadata defines column ID to be an Integer type.

What exactly does the running index identify, the respondent, the questionnaire, a submission, or a record?

The assignment mantions explicitly an framework that allaws resubmissions from the same participant within the the same servey timeline, which means the system must have a key per a single person within a single survey, and the only one can provide that in the data is an ID column.

In [33]:
# minus 1 column name record
len(data.ID) - 1

200

Conclussion: The data analysis confirms the 200 records of surveys

### Analysis of YeshuvAvoda column

Was the job location location was obtained throgh predifined categories or filled in as a free text?
The metadata states that it is stored in a text type field, but there is no mention of haw it was obtained.
From abserving the data it is clear that most likely come from dropdawn list or chousen category becouse the only iregulaarities in the column are bouth valid verssions of the same city, just one older verssion and the other is from newear national regestery verssion. 
Most likely bouth verssions still remained, so I would raise this issue with the relavent stakeholders and peers.

In [34]:
# missing values
data.YeshuvAvoda[(data.YeshuvAvoda.empty) or (data.YeshuvAvoda == None)]

Series([], Name: YeshuvAvoda, dtype: object)

Conclussion: The data analysis confirms that the fieald is a mandotory field which is also aligngs perfectly with my initial asamption.

### Analysis of ShemAvoda column

The metadata defines ShemAvoda as the name of the participant’s workplace.

How many records are empty, how many records are distinct relative to records that not empty, haw match records contain a Latin letters, and haw many records contain a mix of Hebrew and English letters in one cell?


In [35]:
shem_avoda = data.loc[data["ID"].notna(), "ShemAvoda"].astype("string").str.strip().replace("", pd.NA)

print("Empty records:", shem_avoda.isna().sum())
print("Distinct values among nonempty records:", shem_avoda.nunique(), "of", shem_avoda.count())

has_hebrew = shem_avoda.str.contains(r"[א-ת]", na=False)
has_latin = shem_avoda.str.contains(r"[A-Za-z]", na=False)
print("Records containing Hebrew letters:", has_hebrew.sum())
print("Records containing Latin letters:", has_latin.sum())
print("Records containing both Hebrew and Latin letters:", shem_avoda[has_hebrew & has_latin == True])

Empty records: 0
Distinct values among nonempty records: 82 of 200
Records containing Hebrew letters: 187
Records containing Latin letters: 14
Records containing both Hebrew and Latin letters: 139    יועצת IT עצמאית
Name: ShemAvoda, dtype: string


Conclusion: None of the 200 workplace records is empty. There are 82 distinct values among the 200 nonempty records (41%). Fourteen records contain Latin letters, and one of those also contains Hebrew letters in the same cell.

In [38]:
# Set the minimum cosine similarity a match must exceed.
ALPHA = 0.50

# Select each full ShemAvoda value containing at least one Latin letter.
latin_cells = shem_avoda.loc[has_latin]

# Collect the distinct Hebrew-only values already present in ShemAvoda.
hebrew_names = sorted(shem_avoda.loc[has_hebrew & ~has_latin].unique())

# Copy the survey ID and original workplace text for each selected row.
review = data.loc[latin_cells.index, ["ID", "ShemAvoda"]].copy()

# Encode the full selected values and the Hebrew-only values with LaBSE.
vectors = model.encode(latin_cells.tolist() + hebrew_names, batch_size=8, normalize_embeddings=True, convert_to_numpy=True, show_progress_bar=False)

# Compute cosine similarity between each selected value and every Hebrew-only value.
similarities = vectors[:len(latin_cells)] @ vectors[len(latin_cells):].T

# Find the highest-scoring Hebrew-only value for each selected row.
best_indices = similarities.argmax(axis=1)

# Record the highest cosine similarity for each selected row.
review["Cosine"] = similarities[np.arange(len(latin_cells)), best_indices]

# Align the highest-scoring Hebrew-only values with their survey rows.
best_hebrew = pd.Series(np.asarray(hebrew_names)[best_indices], index=review.index, dtype="string")

# Replace the full workplace text when the score exceeds ALPHA and keep the original text otherwise.
review["Transformed"] = best_hebrew.where(review["Cosine"].gt(ALPHA), review["ShemAvoda"])

# Write the transformed text into ShemAvoda in a copy of the survey.
transformed_data = data.copy()
transformed_data.loc[review.index, "ShemAvoda"] = review["Transformed"]

# Display the original text, cosine scores, and transformed text.
display(review)

,ID,ShemAvoda,Cosine,Transformed
9,10.000000,EMILIO,0.428944,EMILIO
14,15.000000,NVIDIA,0.554522,נבידיה
37,38.000000,INTEL,0.752430,אינטל
46,47.000000,NVIDIA,0.554522,נבידיה
55,56.000000,COGNYTE,0.340869,COGNYTE
72,73.000000,EMILIO,0.428944,EMILIO
77,78.000000,NVIDIA,0.554522,נבידיה
100,101.000000,INTEL,0.752430,אינטל
103,104.000000,EMILIO,0.428944,EMILIO
109,110.000000,NVIDIA,0.554522,נבידיה


Which Hebrew workplace value does LaBSE rank first for each standalone name written in Latin letters? Adjust `alpha` below to set the minimum cosine similarity a candidate must exceed to be reviewed.


### Analysis of ShemMachlaka column

The metadata defines `ShemMachlaka` as the department or division where the participant works.

How many department names are blank? Do any of those records describe a department activity?


In [11]:
department_name_missing = text_values("ShemMachlaka").isna()
display(survey.loc[survey.ID.eq(1), ["ID", "ShemAvoda", "ShemMachlaka"]])
print("Blank names:", department_name_missing.sum())
print("Activity supplied with a blank name:",
      (department_name_missing & text_values("SugMachlaka").notna()).sum())


,ID,ShemAvoda,ShemMachlaka
0,1.000000,"למ""ס",מיקרו כלכלה


Blank names: 64
Activity supplied with a blank name: 0


**Conclusion**

I found 64 blank department names. All 64 also had a blank department activity. The records show that neither answer was supplied. They give no reason for the omission.


### Analysis of SugMachlaka column

The metadata defines `SugMachlaka` as the department’s main activity.

What does the activity in record 1 add to the department name? Which preceding answers are available when the activity is recorded as `כנ״ל`?


In [12]:
reference_pattern = r"""כנ["״']?ל"""
def is_reference(field):
    return text_values(field).str.fullmatch(reference_pattern, na=False)

display(survey.loc[survey.ID.eq(1) | is_reference("SugMachlaka"),
                   ["ID", "SugAvoda", "ShemMachlaka", "SugMachlaka"]])


,ID,SugAvoda,ShemMachlaka,SugMachlaka
0,1.000000,"איסוף, עיבוד ופרסום נתונים",מיקרו כלכלה,טיפול בנתוני מיקרו
2,3.000000,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל"
33,34.000000,תיקון קווי מים,תמיכה בלקוחות,"כנ""ל"
65,66.000000,מתן שירות למשתמשים באתר GOV,תמיכה בלקוחות,"כנ""ל"
96,97.000000,הספקת מים,הנדסה,"כנ""ל"
128,129.000000,קבלת קהל,תמיכה בלקוחות,"כנ""ל"
164,165.000000,קבלת קהל,תמיכה בלקוחות,"כנ""ל"
186,187.000000,תיקון קווי חשמל,תמיכה בתקלות,"כנ""ל"
192,193.000000,קבלת קהל,תמיכה בלקוחות,"כנ""ל"


**Conclusion**

In record 1, `טיפול בנתוני מיקרו` explains what the department does. Where the participant wrote `כנ״ל`, the workbook provides no rule for choosing the answer it refers to. I left those references out of the model text.


### Analysis of EzoAvoda column

The metadata defines `EzoAvoda` as the work the participant does at the workplace.

For records with no work description, what task or title information is available?


In [13]:
display(survey.loc[text_values("EzoAvoda").isna(), ["ID", "TeurPeula", "TeurTafkid"]])


,ID,TeurPeula,TeurTafkid
44,45.000000,nan,nan
107,108.000000,nan,nan
141,142.000000,nan,nan
163,164.000000,nan,nan
165,166.000000,nan,nan
170,171.000000,nan,מנהלת משאבי אנוש
192,193.000000,nan,nan


**Conclusion**

Seven records have no work description. I kept the work description as the main source of occupational text and kept tasks and titles as separate sources of evidence.


### Analysis of TeurPeula column

The metadata defines `TeurPeula` as the main tasks the participant performs.

What does the task description add in record 3? How many task answers are recorded as `כנ״ל`?


In [14]:
display(survey.loc[survey.ID.eq(3), ["ID", "EzoAvoda", "TeurPeula"]])
print("Task references:", is_reference("TeurPeula").sum())


,ID,EzoAvoda,TeurPeula
2,3.000000,נותנת תמיכה ללקוחות שמבקשים עזרה,מסבירה איך להשתמש בשירותים ממשלתיים באתר האינטרנט


Task references: 12


**Conclusion**

In record 3, the work description mentions customer support, and the tasks explain that the participant helps people use government services online. I kept the task text and left the 12 unresolved references out of the model text.


### Analysis of TeurTafkid column

The metadata asks for a job title if the participant has one.

How many titles are blank? What work is reported when the title is recorded as `0` or `כנ״ל`?


In [15]:
title = text_values("TeurTafkid")
print("Blank titles:", title.isna().sum())
display(survey.loc[title.eq("0").fillna(False) | is_reference("TeurTafkid"),
                   ["ID", "EzoAvoda", "TeurTafkid"]])


Blank titles: 172


,ID,EzoAvoda,TeurTafkid
1,2.000000,טכנאית רדיותרפיה,"כנ""ל"
32,33.000000,טכנאית רדיותרפיה,"כנ""ל"
38,39.000000,"מהנדס תוכנה, מנהל צוות פיתוח",0
64,65.000000,טכנאית רדיותרפיה,"כנ""ל"
95,96.000000,טכנאית רדיותרפיה,"כנ""ל"


**Conclusion**

I found 172 blank titles. In record 39, the participant describes software engineering and team management and gives `0` as the title. I kept the original answer and left `0` and unresolved references out of the title text.


### Analysis of MenahelEtMi column

The metadata defines code 1 as managing employees, 2 as managing managers, and 3 as managing both.

How many survey records contain each code or a blank? What value appears on the worksheet row without a survey ID?


In [16]:
display(data.loc[data.ID.isna(), ["ID", "MenahelEtMi"]])
display(survey.MenahelEtMi.value_counts(dropna=False).rename("records"))
management_level = survey.MenahelEtMi.fillna(0).astype(int)


,ID,MenahelEtMi
200,nan,1.725000


MenahelEtMi
NaN    168
3.0     13
2.0     11
1.0      8
Name: records, dtype: int64

**Conclusion**

I found 32 answers and 168 blanks. The value 1.725 is in cell J202 on a row without a survey ID. I excluded that row. In the derived management field, I used 0 for the blanks and treated the codes as levels, as specified in the workbook interpretation.


### Analysis of MaamadAvoda column

The metadata defines seven employment arrangements.

Which of the seven codes appear in the survey, and how often?


In [17]:
display(survey.MaamadAvoda.value_counts(dropna=False).sort_index().rename("records"))


MaamadAvoda
1.0    177
2.0      1
4.0     10
6.0     10
NaN      2
Name: records, dtype: int64

**Conclusion**

I found four arrangements in the sample. Each code names an employment arrangement, so I would use it as a category in a model. The codes also determine who is eligible to answer the salary-source question.


### Analysis of MakorSachar column

The metadata limits the salary-source question to employees and cooperative members.

Which records give a salary source despite an ineligible employment status? How many records lack the status needed to check eligibility?


In [18]:
status = survey.MaamadAvoda
salary = survey.MakorSachar
display(pd.crosstab(status.astype("Int64").astype("string").fillna("Missing"),
                    salary.astype("Int64").astype("string").fillna("Missing")))
salary_conflict = status.isin([2, 3, 4, 6, 7]) & salary.notna()
display(survey.loc[salary_conflict, ["ID", "MaamadAvoda", "MakorSachar"]])


MakorSachar,1,2,3,4,Missing
MaamadAvoda,,,,,
1,136,26,11,4,0
2,1,0,0,0,0
4,7,0,0,1,2
6,0,0,0,0,10
Missing,0,0,0,0,2


,ID,MaamadAvoda,MakorSachar
33,34.000000,4.000000,4.000000
48,49.000000,4.000000,1.000000
111,112.000000,4.000000,1.000000
139,140.000000,4.000000,1.000000
165,166.000000,4.000000,1.000000
175,176.000000,4.000000,1.000000
186,187.000000,4.000000,1.000000
189,190.000000,2.000000,1.000000
197,198.000000,4.000000,1.000000


**Conclusion**

Nine participants supplied a salary source even though their employment status made them ineligible. Two records had no employment status. I kept the original answers and distinguished these cases from expected blanks.


### Analysis of TeudaGvoha column

The metadata defines `TeudaGvoha` as the highest certificate or degree the participant received. Code 8 means another certificate and code 9 means no certificate.

Which certificate codes appear in the survey?


In [19]:
display(survey.TeudaGvoha.value_counts(dropna=False).sort_index().rename("records"))


TeudaGvoha
1.0     3
2.0    12
3.0    57
4.0    58
5.0    25
6.0    39
7.0     4
NaN     2
Name: records, dtype: int64

**Conclusion**

The survey contains codes 1 through 7. The metadata also defines 8 as another certificate and 9 as no certificate. I would treat these codes as categories because their numerical order does not consistently represent qualification level.


### Analysis of shnotlimud column

The metadata defines `shnotlimud` as years of schooling. The workbook interpretation proposes a range of 8 to 80 years.

Which recorded values fall outside this range or are fractional?


In [20]:
school_years = survey.shnotlimud
invalid_school_years = school_years.notna() & (
    ~school_years.between(8, 80) | school_years.mod(1).ne(0)
)
display(school_years.agg(["count", "min", "max"]))
display(survey.loc[invalid_school_years, ["ID", "shnotlimud"]])


count    198.0
min        7.0
max       22.0
Name: shnotlimud, dtype: float64

,ID,shnotlimud
33,34.000000,7.000000
192,193.000000,7.000000


**Conclusion**

Two recorded values fall below the proposed minimum of eight years. I flagged them for review and kept the original values.


### Analysis of Gil column

The metadata defines `Gil` as the participant’s age. The workbook interpretation proposes a range of 18 to 120 years.

Which recorded ages fall outside this range or are fractional?


In [21]:
age = survey.Gil
invalid_age = age.notna() & (~age.between(18, 120) | age.mod(1).ne(0))
display(age.agg(["count", "min", "max"]))
display(survey.loc[invalid_age, ["ID", "Gil"]])


count    198.0
min       22.0
max       78.0
Name: Gil, dtype: float64

,ID,Gil


**Conclusion**

The 198 recorded ages are whole numbers from 22 to 78. All pass the proposed check.


### Analysis of TarichHathala column

The metadata defines `TarichHathala` as the date the participant started the role.

How many distinct start dates are recorded? Are there values that cannot be read as dates?


In [22]:
role_start = pd.to_datetime(survey.TarichHathala, errors="coerce")
display(role_start.agg(["count", "min", "max", "nunique"]))
display(survey.loc[survey.TarichHathala.notna() & role_start.isna(), ["ID", "TarichHathala"]])


count                      200
min        1999-06-06 00:00:00
max        2026-06-06 00:00:00
nunique                     42
Name: TarichHathala, dtype: object

,ID,TarichHathala


**Conclusion**

I found 42 distinct start dates in 200 records. To calculate tenure at the time of the survey, I would need the survey reference date. I kept the start dates and left tenure out of the model.


### Analysis of SemelAnafSofi column

The metadata defines `SemelAnafSofi` as the four-digit industry code of the workplace.

How does Excel display numeric industry codes, and which codes contain `X`?


In [23]:
from openpyxl import load_workbook

workbook = load_workbook(io.BytesIO(raw), read_only=True)
sheet = workbook["Survey Dataset"]
code_fields = ["SemelAnafSofi"]
headers = next(sheet.values)
code_positions = [headers.index(field) for field in code_fields]
def confirm_code_format(sheet, position):
    for row in sheet.iter_rows(min_row=2, max_row=len(data) + 1):
        if row[position].data_type == "n" and row[position].value is not None:
            assert row[position].number_format == "0000"
confirm_code_format(sheet, code_positions[0])
workbook.close()

def displayed_codes(field):
    values = text_values(field).str.upper()
    numeric = values.str.fullmatch(r"[0-9]{1,4}", na=False)
    return values.where(~numeric, values.str.zfill(4))


industry = displayed_codes("SemelAnafSofi")
display(industry[industry.str.contains("X", na=False)].value_counts().rename("records"))


SemelAnafSofi
XXXX    6
851X    1
852X    1
3XXX    1
Name: records, dtype: Int64

**Conclusion**

Excel uses four positions for numeric industry codes, so I kept any leading zero and retained the recorded `X` positions. I left industry out of the model because the assignment does not say when that code becomes available.


### Analysis of SemelMishlachSofi column

The assignment identifies `SemelMishlachSofi` as the manually assigned occupation code and uses `X` for unresolved positions.

How does Excel display the numeric codes? How many records have a complete code, and which incomplete patterns occur?


In [24]:
workbook = load_workbook(io.BytesIO(raw), read_only=True)
sheet = workbook["Survey Dataset"]
confirm_code_format(sheet, next(sheet.values).index("SemelMishlachSofi"))
workbook.close()

TARGET = "SemelMishlachSofi"
target = displayed_codes(TARGET)
complete = target.str.fullmatch(r"[0-9]{4}", na=False)
partial = target.str.fullmatch(r"[0-9]*X+", na=False) & target.str.len().eq(4).fillna(False)
display(target[partial].value_counts().rename("records"))
display(survey.loc[~(complete | partial), ["ID", TARGET]])
print("Complete codes:", complete.sum(), "| Incomplete codes:", partial.sum())


SemelMishlachSofi
1XXX    4
XXXX    4
335X    1
Name: records, dtype: Int64

,ID,SemelMishlachSofi


Complete codes: 191 | Incomplete codes: 9


**Conclusion**

There are 191 complete codes and nine with unresolved positions. Use the complete codes for training and evaluation. Set aside the nine X-coded records for suggestions and check each suggestion against any prefix the coder supplied.


## Classification and evaluation

How well do duties, workplace context, and both together predict the recorded occupation code? Compare them with a majority baseline. Keep identical occupational descriptions in the same fold and fit each vectorizer on training rows.


In [25]:
from sklearn.pipeline import make_pipeline

role_fields = ["EzoAvoda", "TeurPeula", "TeurTafkid"]
context_fields = ["YeshuvAvoda", "ShemAvoda", "SugAvoda", "ShemMachlaka", "SugMachlaka"]
descriptions = survey[role_fields + context_fields].astype("string").apply(
    lambda values: values.str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
)
descriptions = descriptions.replace(r"^(?:כנ[\"״']?ל|חסוי|לא יכול לפרט|)$", pd.NA, regex=True)
descriptions["TeurTafkid"] = descriptions["TeurTafkid"].replace("0", pd.NA)
groups = descriptions[role_fields].fillna("").agg(" | ".join, axis=1)
views = {"role": role_fields, "context": context_fields, "combined": role_fields + context_fields}
model_text = {
    view: descriptions[fields].fillna("").agg(" ".join, axis=1)
    for view, fields in views.items()
}
labels = target[complete]
folds = min(5, groups[complete].nunique())
splits = list(GroupKFold(n_splits=folds).split(labels, labels, groups[complete]))


In [26]:
models = {
    "Majority baseline": DummyClassifier(strategy="most_frequent"),
    "Word TF-IDF logistic regression": make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), smooth_idf=False, norm=None),
        LogisticRegression(max_iter=1500, class_weight="balanced", random_state=SEED),
    ),
    "Character TF-IDF SVC": make_pipeline(
        TfidfVectorizer(analyzer="char", ngram_range=(2, 5), smooth_idf=False, norm=None),
        LinearSVC(class_weight="balanced", random_state=SEED),
    ),
    "Character TF-IDF naive Bayes": make_pipeline(
        TfidfVectorizer(analyzer="char", ngram_range=(2, 4), smooth_idf=False, norm=None),
        ComplementNB(alpha=0.5),
    ),
}
scores, predictions = [], []
for view, texts in model_text.items():
    training_text = texts[complete]
    for name, model in models.items():
        held_out = pd.Series(index=labels.index, dtype="object")
        unseen = 0
        for train, test in splits:
            fitted = clone(model).fit(training_text.iloc[train], labels.iloc[train])
            held_out.iloc[test] = fitted.predict(training_text.iloc[test])
            unseen += (~labels.iloc[test].isin(labels.iloc[train])).sum()
        scores.append({
            "view": view, "model": name, "accuracy": accuracy_score(labels, held_out),
            "macro_f1": f1_score(labels, held_out, average="macro", zero_division=0),
            "unseen_label_rows": int(unseen),
        })
        predictions.append(pd.DataFrame({
            "ID": survey.loc[complete, "ID"], "view": view, "model": name,
            "actual": labels, "predicted": held_out,
        }))
results = pd.DataFrame(scores).sort_values(["macro_f1", "accuracy"], ascending=False)
display(results)


,view,model,accuracy,macro_f1,unseen_label_rows
10,combined,Character TF-IDF SVC,0.345550,0.153644,106
9,combined,Word TF-IDF logistic regression,0.303665,0.148818,106
6,context,Character TF-IDF SVC,0.314136,0.143114,106
5,context,Word TF-IDF logistic regression,0.282723,0.135490,106
11,combined,Character TF-IDF naive Bayes,0.319372,0.132705,106
7,context,Character TF-IDF naive Bayes,0.314136,0.130401,106
3,role,Character TF-IDF naive Bayes,0.277487,0.124479,106
2,role,Character TF-IDF SVC,0.225131,0.109225,106
1,role,Word TF-IDF logistic regression,0.109948,0.070823,106
0,role,Majority baseline,0.020942,0.000812,106


## Review of classification decisions

Which suggestions for X-coded records disagree with a known prefix? Which records share a description but have conflicting complete codes, or have no substantive description of duties? Use the highest-scoring text configuration to generate the suggestions.


In [27]:
best = results.loc[results.model.ne("Majority baseline")].iloc[0]
best_text = model_text[best["view"]]
final_model = clone(models[best["model"]]).fit(best_text[complete], labels)
review = pd.DataFrame({"ID": survey.ID, TARGET: target})
review["suggestion"] = pd.Series(pd.NA, index=survey.index, dtype="string")
if partial.any():
    review.loc[partial, "suggestion"] = final_model.predict(best_text[partial])
review["prefix_conflict"] = False
review.loc[partial, "prefix_conflict"] = [
    not suggestion.startswith(code.rstrip("X"))
    for suggestion, code in zip(review.loc[partial, "suggestion"], target[partial])
]
conflicting_groups = labels.groupby(groups[complete]).nunique()
review["conflicting_complete_codes"] = groups.map(conflicting_groups).gt(1) & groups.ne(" |  | ")
review["no_duties_recorded"] = descriptions[role_fields].isna().all(axis=1)
review["needs_human_review"] = (
    ~complete | review["conflicting_complete_codes"] | review["no_duties_recorded"]
)
display(review.loc[review.needs_human_review])
assert survey.equals(data.loc[data.ID.notna()])


,ID,SemelMishlachSofi,suggestion,prefix_conflict,conflicting_complete_codes,no_duties_recorded,needs_human_review
1,2.000000,3311,,False,True,False,True
10,11.000000,3513,,False,True,False,True
17,18.000000,1XXX,5414,True,False,True,True
32,33.000000,3211,,False,True,False,True
44,45.000000,XXXX,5131,False,False,True,True
49,50.000000,1XXX,5414,True,False,True,True
53,54.000000,2131,,False,True,False,True
64,65.000000,3211,,False,True,False,True
73,74.000000,3513,,False,True,False,True
80,81.000000,1XXX,5120,True,False,True,True


In [28]:
output = Path("cbs_third_generation_results")
output.mkdir(exist_ok=True)
results.to_csv(output / "model_ablation.csv", index=False)
review.to_csv(output / "human_review_queue.csv", index=False, encoding="utf-8-sig")
pd.concat(predictions, ignore_index=True).to_csv(output / "held_out_predictions.csv", index=False)
manifest = {
    "source_sha256": SHA, "rows": len(survey), "columns": len(survey.columns),
    "full_label_rows": len(labels), "cv_folds": folds, "best": best.to_dict(),
}
(output / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")


345